# TourismGPT User Interface Using Gradio

## Overview

This notebook demonstrates the deployment of TourismGPT through an interactive web interface built using Gradio. After fine-tuning the Phi-3 Mini model and integrating the Retrieval-Augmented Generation (RAG) pipeline, Gradio provides a simple and user-friendly interface for interacting with the chatbot.

The interface allows users to ask tourism-related questions and receive context-aware responses generated by the fine-tuned language model using relevant information retrieved from the FAISS knowledge base.

# TourismGPT — Chat UI Demo
Gradio interface for the fine-tuned Phi-3 Mini + Wikivoyage RAG pipeline.

**Prerequisites:** Run `rag_pipeline.ipynb` first to build the FAISS index and save the adapter to Drive.

## Environment Setup

The required libraries are installed to load the fine-tuned model, retrieve tourism documents using the FAISS index, and deploy the chatbot using Gradio.

These libraries enable seamless integration between the language model, retrieval pipeline, and user interface.

In [ ]:
# CELL 1 — Install dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install sentence-transformers faiss-gpu gradio

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-3mi7ol9x/unsloth_893b99bec23545d9b134d4fa6b100221
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-3mi7ol9x/unsloth_893b99bec23545d9b134d4fa6b100221
  Resolved https://github.com/unslothai/unsloth.git to commit 278e9e7921a56c603a3384e1bdc8562c4e354858
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 137.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 135.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 25.9 MB/s eta 0:00:00
 

In [ ]:
# CELL 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import Required Libraries

This section imports the Python libraries required for chatbot deployment.

The imported modules support model loading, tokenizer initialization, document retrieval, vector search, and creation of the interactive Gradio interface.

In [ ]:
# CELL 3 — Imports and config
import bz2, re, os, pickle
import numpy as np
import torch
from xml.etree import ElementTree as ET

ADAPTER_DIR  = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/tourism_gpt_adapter"
DRIVE_INDEX  = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/wikivoyage_index"
EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K        = 3
MAX_SEQ_LEN  = 2048
MAX_NEW_TOKENS = 400

print("Config ready ✓")

Config ready ✓


## Loading the Tourism Knowledge Base

The FAISS vector database containing tourism document embeddings is loaded during application startup.

For every user query, the chatbot searches this knowledge base to retrieve the most relevant tourism information, which is then provided to the language model as additional context before response generation.

In [ ]:
# CELL 4 — Load FAISS index + chunks from Drive
import faiss
from sentence_transformers import SentenceTransformer

print("Loading embedding model ...")
embedder = SentenceTransformer(EMBED_MODEL)

print("Loading FAISS index ...")
index = faiss.read_index(f"{DRIVE_INDEX}/wikivoyage.index")
with open(f"{DRIVE_INDEX}/chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

print(f"Index: {index.ntotal} vectors  |  Chunks: {len(all_chunks)}")

Loading embedding model ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading FAISS index ...
Index: 18774 vectors  |  Chunks: 18774


## Loading the Fine-Tuned Language Model

The previously trained Phi-3 Mini model and tokenizer are loaded into memory for inference.

Instead of retraining the model, the saved adapter weights are restored, allowing the chatbot to generate tourism-specific responses efficiently while preserving the knowledge learned during fine-tuning.

In [ ]:
# CELL 5 — Load fine-tuned Phi-3 + LoRA adapter
from unsloth import FastLanguageModel

print("Loading Phi-3 Mini + LoRA adapter ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)
print("Model ready ✓")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Phi-3 Mini + LoRA adapter ...
==((====))==  Unsloth 2026.7.5: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.7.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model ready ✓


## Query Processing Workflow

Whenever a user submits a tourism-related question, the chatbot performs the following sequence of operations:

1. Accepts the user's input.
2. Converts the query into a semantic embedding.
3. Searches the FAISS vector database for relevant tourism documents.
4. Retrieves the most relevant context.
5. Combines the retrieved context with the user's query.
6. Sends the enriched prompt to the fine-tuned Phi-3 Mini model.
7. Returns a context-aware response through the Gradio interface.

This Retrieval-Augmented Generation (RAG) workflow improves response accuracy while reducing hallucinations.

In [ ]:
# CELL 6 — RAG functions
def retrieve(query, k=TOP_K):
    q_emb = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k)
    return [{**all_chunks[idx], "score": float(score)}
            for score, idx in zip(scores[0], indices[0])]

def ask_rag(question):
    chunks = retrieve(question)
    context = "\n\n".join(
        f"[{i}] {c['title']}: {c['text']}" for i, c in enumerate(chunks, 1)
    )
    prompt = (
        f"<|user|>\n"
        f"You are TourismGPT, an expert travel assistant. "
        f"Use the context below to give a helpful, specific answer. "
        f"If the context does not cover the question, answer from your training knowledge.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}<|end|>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    max_input = MAX_SEQ_LEN - MAX_NEW_TOKENS
    if inputs["input_ids"].shape[1] > max_input:
        inputs = {k: v[:, -max_input:] for k, v in inputs.items()}
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = MAX_NEW_TOKENS,
                temperature    = 0.7,
                top_p          = 0.9,
                do_sample      = True,
                pad_token_id   = tokenizer.eos_token_id,
            )
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    finally:
        # Release the input tensors / KV cache promptly so a slow GPU (T4, 4-bit
        # Phi-3) doesn't accumulate memory across turns and OOM mid-request —
        # an OOM here crashes the process and the Gradio client sees an HTML
        # error page instead of JSON ("Unexpected token '<'").
        del inputs
        torch.cuda.empty_cache()
    sources = list({c["title"] for c in chunks})
    return answer, sources

print("RAG functions ready ✓")

RAG functions ready ✓


## Creating the Interactive User Interface

Gradio provides a lightweight web application that enables users to interact with TourismGPT without requiring programming knowledge.

The interface consists of a text input field for user questions and a response area where the chatbot displays tourism-related answers generated by the fine-tuned model.

This interface demonstrates how industry-specific Large Language Models can be deployed as practical conversational AI applications.

In [ ]:
# CELL 7 — Launch Gradio chat UI
import gradio as gr
import traceback

EXAMPLE_QUESTIONS = [
    "Plan a 5-day itinerary for Tokyo on a budget.",
    "Compare Bali vs Thailand for a honeymoon.",
    "What is the estimated budget for a week in Paris?",
    "What cultural customs should I know before visiting Japan?",
    "What are the best things to do in Rome?",
    "Is Bali safe for solo female travellers?",
]

def chat(message, history):
    if not message.strip():
        return history, "", ""
    history = history or []
    history.append({"role": "user", "content": message})
    try:
        answer, sources = ask_rag(message)
    except Exception:
        traceback.print_exc()
        history.append({"role": "assistant", "content": "Sorry, something went wrong generating a response. Please try again."})
        return history, "", ""
    history.append({"role": "assistant", "content": answer})
    source_text = "**Sources:** " + " · ".join(f"`{s}`" for s in sources)
    return history, "", source_text

with gr.Blocks(title="TourismGPT") as demo:
    gr.Markdown(
        "# ✈️ TourismGPT\n"
        "**Fine-tuned Phi-3 Mini + Wikivoyage RAG** — Ask anything about travel!"
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=460, label="TourismGPT")

            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Ask a travel question ...",
                    show_label=False,
                    scale=5,
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)
            sources_box = gr.Markdown("", label="Sources")
            gr.ClearButton([chatbot, msg, sources_box], value="Clear chat")

        with gr.Column(scale=1):
            gr.Markdown("### Try these questions")
            for q in EXAMPLE_QUESTIONS:
                gr.Button(q, size="sm").click(
                    fn=lambda x=q: x, outputs=msg
                )

    send_btn.click(chat, [msg, chatbot], [chatbot, msg, sources_box])
    msg.submit(chat, [msg, chatbot], [chatbot, msg, sources_box])

demo.queue(default_concurrency_limit=1)
demo.launch(share=True, debug=True, theme=gr.themes.Soft())

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f7dc0c98403bc8fcac.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

# Conclusion

The Gradio interface completes the TourismGPT system by providing an intuitive platform for user interaction.

By combining the fine-tuned Phi-3 Mini model with Retrieval-Augmented Generation (RAG), the chatbot delivers reliable, context-aware tourism assistance through an accessible web interface.

This notebook demonstrates the final deployment stage of the project, transforming the underlying machine learning pipeline into a practical conversational AI application suitable for real-world tourism support.